<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2012/CrossValidationforKinKNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Required Assignment 12.1: Identifying the Best K

This activity focuses on identifying the "best" number of neighbors that optimize the accuracy of a `KNearestNeighbors` estimator. The ideal number of neighbors will be selected through cross-validation and a grid search over the `n_neighbors` parameter.  Again, prior to building the model, you will want to scale the data in a `Pipeline`.

**Expected Time: 60 Minutes**

**Total Points: 50**

#### Index

- [Problem 1](#Problem-1)
- [Problem 2](#Problem-2)
- [Problem 3](#Problem-3)
- [Problem 4](#Problem-4)
- [Problem 5](#Problem-5)
- [Problem 6](#Problem-6)


In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


### The Dataset

Again, you will use the credit default dataset to predict default -- yes or no.  The data is loaded and split into train and test sets for you below.  You will again build a column transformer to encode the `student` feature.  Note that scikit-learn handles a string target features in the `KNeighborsClassifier`, and we do not need to encode this column.

In [106]:
df = pd.read_csv('/content/sample_data/default.csv', index_col=0)

In [107]:
df.tail(2)

,default,student,balance,income
9999,No,No,1569.009053,36669.112365
10000,No,Yes,200.922183,16862.952321


In [108]:
X= df.drop(columns={"default"})
y= df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.25,random_state=42)

In [109]:
X_test

,student,balance,income
6253,No,1435.662933,31507.089277
4685,No,771.789347,42139.070269
1732,No,0.000000,21809.218509
4743,No,113.571264,32803.832648
4522,No,1358.132472,49903.597081
...,...,...,...
4863,No,306.459196,36812.626047
7026,Yes,1947.072679,13157.956557
7648,No,0.000000,33769.604866
7162,Yes,470.107181,16014.113311


[Back to top](#-Index)

### Problem 1

#### Baseline for Models

**5 Points**

Before starting the modeling process, you should have a baseline to determine whether your model is any good.

Consider the `default` column of `df`. Perform a `value_counts` operation with the argument `normalize` equal to `True`.

What would the accuracy of such a classifier be?  Enter your answer as a float to `baseline` below.



In [110]:
### GRADED
baseline = ''
# YOUR CODE HERE
print(df['default'].value_counts(normalize=True))
baseline=0.9667
# Answer check
print(baseline)

default
No     0.9667
Yes    0.0333
Name: proportion, dtype: float64
0.9667


[Back to top](#-Index)

### Problem 2

#### Column transforms and KNN

**10 Points**

Use the `make_column_transformer` to create a column `transformer`. Inside the `make_column_transformer` specify an instance of the `OneHotEncoder` transformer from scikit-learn. Inside `OneHotEncoder` set `drop` equal to `'if_binary'`. Apply this transformation to the `student` column. On the `remainder` columns, apply a `StandardScaler()` transformation.


Next, build a `Pipeline` named `knn_pipe` with  steps `transform` and `knn`. Set `transform` equal to `transformer` and `knn` equal to `KNeighborsClassifier()`. Be sure to leave all the settings in `knn` to default.  

In [111]:
### GRADED
transformer = ''
knn_pipe = ''
# YOUR CODE HERE
cat_cols= ['student']
transformer= make_column_transformer((OneHotEncoder(drop='if_binary'),cat_cols), remainder=StandardScaler())
knn_pipe= Pipeline(steps= [("transform",transformer), ("knn",KNeighborsClassifier())])

# Answer check
knn_pipe


Pipeline(steps=[('transform',
                 ColumnTransformer(remainder=StandardScaler(),
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['student'])])),
                ('knn', KNeighborsClassifier())])

[Back to top](#-Index)

### Problem 3

#### Parameter grid

**10 Points**

Now that your pipeline is ready, you are to construct a parameter grid to search over.  Consider two things:

- You will not be able to predict on a test dataset where `n_neigbors > len(test_data)`.  This will limit our upper bound on `k`.  In this example, too high a `k` will slow down the computation, so only consider `k = [1, 3, 5, ..., 21]`.
- Ties in voting are decided somewhat arbitrarily and for speed and clarity you should consider only odd values for the number of neighbors

Creating a dictionary called `params` that specifies hyperparameters for the KNN classifier.

- The key of your dictionary will be `knn__n_neighbors`
- The values in your dictionary will be `list(range(1, 22, 2))`



In [112]:
### GRADED
params = ''
# YOUR CODE HERE
k=list()
for i in range(0,11):
    k.append(i*2 +1 )
#k[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
params= {"knn__n_neighbors":list(range(1, 22, 2))}
#params= {"knn__n_neighbors":k}


# Answer check
#list(params.values())[0]


[Back to top](#-Index)

### Problem 4

#### Grid search `k`

**10 Points**

- Use `GridSearchCV` with the `knn_pipe` and `param_grid` equal to `params`. Assign the result to `knn_grid`.
- Use the `fit` function on `knn_grid` to train your model on `X_train` and `y_train`.
- Retrieve the best value for the hyperparameter `k` from the `best_params_` attribute of the grid search object `knn_grid`. Assign the result to `best_k`.
- Use the `score` function to calculate the accuracy of the `knn_grid` classifier on a test dataset. Assign your best models accuracy on the test data as a float to `best_acc`



In [113]:
### GRADED
# GridSearchCV(pipeline_model ,params, score = 'accuracy_score', )
knn_grid = ''
best_k = ''
best_acc = ''

# YOUR CODE HERE
knn_grid = GridSearchCV(knn_pipe, params)# by default uses 5-fold stratified CV
knn_grid.fit(X_train, y_train)
best_est= knn_grid.best_estimator_
best_k= list(knn_grid.best_params_.values())[0]
y_preds = knn_grid.best_estimator_.predict(X_test)
best_acc = accuracy_score(y_preds,y_test)
print(knn_grid.best_estimator_.score(X_test, y_test))

# Answer check
print(best_acc)
print(best_k)

0.9708
0.9708
11


In [114]:
### BEGIN HIDDEN TESTS
params_ = {'knn__n_neighbors': list(range(1, 22, 2))}
knn_grid_ = GridSearchCV(knn_pipe, param_grid=params_)
knn_grid_.fit(X_train, y_train)
best_k_ = list(knn_grid_.best_params_.values())[0]
best_acc_ = knn_grid_.score(X_test, y_test)
#
#
#
print("best_k_",best_k_)
print("best_k",best_k)
assert best_k == best_k_
assert best_acc == best_acc_
### END HIDDEN TESTS

best_k_ 11
best_k 11


[Back to top](#-Index)

### Problem 5

#### Other parameters to consider

**10 Points**

The number of neighbors is not the only parameter in the implementation from scikit-learn.  For example, you can also consider different weightings of points based on their distance, change the distance metric, and search over alternative versions of certain metrics like Minkowski.  See the docstring from `KNeighborsClassifier` below.

```
weights : {'uniform', 'distance'} or callable, default='uniform'
    Weight function used in prediction.  Possible values:

    - 'uniform' : uniform weights.  All points in each neighborhood
      are weighted equally.
    - 'distance' : weight points by the inverse of their distance.
      in this case, closer neighbors of a query point will have a
      greater influence than neighbors which are further away.
    - [callable] : a user-defined function which accepts an
      array of distances, and returns an array of the same shape
      containing the weights.
      
===========================

p : int, default=2
    Power parameter for the Minkowski metric. When p = 1, this is
    equivalent to using manhattan_distance (l1), and euclidean_distance
    (l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.
    
```

Create a new parameter grid and consider both weightings as well as `p = [1, 2]`.  Assign this as a dictionary to `params2` below.  

Search over these parameters in your `knn_pipe` with a `GridSearchCV` named `weight_grid` below. Also, consider `n_neighbors` as in [Problem 4](#-Problem-4).  Did your new grid search results perform better than earlier?  Assign this grids accuracy to `weights_acc` below.

In [115]:
def best_score_for_weighted_minkowski_p12(pipe):
  weight_grid =GridSearchCV(pipe, params)
  weight_grid.fit(X_train, y_train)
  weights_acc_pipe = weight_grid.best_estimator_.score(X_test, y_test)
  print (weight_grid.cv_results_)
  return weights_acc_pipe


In [102]:
### GRADED
params2 = ''
weight_grid = ''
weights_acc = ''
# YOUR CODE HERE
weights=['uniform', 'distance']
allacc= list()
p =[1, 2]

params2= {"knn__n_neighbors": k,
          "knn__p": p ,
          "knn__weights":weights,
          }

weight_grid =GridSearchCV(knn_pipe, params2)
weight_grid.fit(X_train, y_train)
weights_acc_pipe = weight_grid.best_estimator_.score(X_test, y_test)


weights_acc =  weights_acc_pipe
# Answer check
print(weights_acc)
print(weight_grid.best_params_)


0.9708
{'knn__n_neighbors': 11, 'knn__p': 2, 'knn__weights': 'uniform'}


In [103]:
params

{'knn__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]}

[Back to top](#-Index)

### Problem 6

#### Further considerations

**5 Points**

When performing your grid search you want to also be sensitive to the amount of parameters you are searching and the number of different models being built.  How many models were constructed in [Problem 5](#-Problem-5)?  Enter your answer as an integer to `ans6` below.  You might use the grids `.cv_results_` attribute to determine this.

In [126]:
### GRADED
ans6 = ''
# YOUR CODE HERE
# default value of cv=5 for grid search and 2 p values (1,2) and 2 weights (uniform and distance) needs to be multiplied by number of paramters = 5
#basically number of param  combinatuon in params2 (2 *2 * 11)
ans6 = len(range(1,22,2)) * 2 * 2 *5

# Answer check
print(ans6)


220
